# Parallel Chains

In [2]:
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from dotenv import load_dotenv
from rich import print
from rich.logging import RichHandler
from rich.table import Table
from rich.console import Console
import time
import os
import logging
import warnings

warnings.filterwarnings("ignore")

logger = logging.getLogger()
logger.setLevel(logging.INFO)

console_handler = RichHandler()
console_handler.setLevel(logging.INFO)

file_handler = logging.FileHandler("app.log", mode="a", encoding="utf-8")
file_handler.setLevel(logging.INFO)

formatter = logging.Formatter(
    "%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"    
)

file_handler.setFormatter(formatter)

logger.addHandler(console_handler)
logger.addHandler(file_handler)

console = Console()

load_dotenv()

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_API_ENDPOINT = os.getenv("AZURE_OPENAI_API_ENDPOINT")
AZURE_OPENAI_API_DEPLOYMENT_4O = os.getenv("AZURE_OPENAI_API_DEPLOYMENT_4O")

llm_openai = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_API_ENDPOINT,
    deployment_name=AZURE_OPENAI_API_DEPLOYMENT_4O,
    openai_api_key=AZURE_OPENAI_API_KEY,
    openai_api_version=AZURE_OPENAI_API_VERSION
)

# TASK - 1 [Prompt]

prompt_template = ChatPromptTemplate.from_messages([
    ("system","You are a movie summarizer"),
    ("human","Please summarize the movie in brief: {input}")
])

# TASK - 2 [LLM]

# TASK - 3 [Str Parser]

str_parser = StrOutputParser()

# TASK - 4 [Custom Runnable]
def dictionary_maker(text:str)->dict:
    return {"text":text}

dictionary_maker_runnable = RunnableLambda(dictionary_maker)

# Parallel Chain 1

# TASK - 1 [Prompt]
linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system","You are a LinkedIn post generator"),
    ("human","Create a post for the following text for LinkedIn: {text}")
])

# TASK - 2 [LLM]

# TASK - 3 [Str Parser]
str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm_openai | str_parser

# Parallel Chain 2

from langchain_core.runnables import RunnableParallel, RunnableLambda

def insta_chain(text:dict):

    text = text["text"]

    # TASK - 1 [Prompt]
    insta_prompt = ChatPromptTemplate.from_messages([
        ("system","You are a LinkedIn post generator"),
        ("human","Create a post for the following text for Instagram: {text}")
    ])

    # TASK - 2 [LLM]
    
    # TASK - 3 [Str Parser]
    str_parser = StrOutputParser()

    chain_insta = insta_prompt | llm_openai | str_parser

    result = chain_insta.invoke(text)

    return result

insta_chain_runnable = RunnableLambda(insta_chain)

# Final Orchestration

final_chain = (
    prompt_template
    | llm_openai
    | str_parser
    | dictionary_maker_runnable
    | RunnableParallel(branches = {"linkedin": chain_linkedin, "instagram": insta_chain_runnable})
)


final_chain.invoke("KGF")




[04/01/26 19:13:16] INFO     HTTP Request: POST                                                     ]8;id=838930;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=669316;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

[04/01/26 19:13:19] INFO     HTTP Request: POST                                                     ]8;id=369390;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=407416;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

                    INFO     HTTP Request: POST                                                     ]8;id=419532;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=986952;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

{'branches': {'linkedin': "💥 Spotlight on #KGF (Kolar Gold Fields) 💥\n\nReleased in 2018, KGF is an Indian action-drama that captivated audiences with its gripping storytelling and powerful performances. The film traces the journey of **Rocky**, a young man from Mumbai driven by ambition and the promise he made to his mother—to become powerful and wealthy.\n\nRocky's quest leads him to the infamous gold mines of Karnataka, where he navigates ruthless criminals, corrupt politicians, and his own moral dilemmas. Through high-octane action and intense drama, KGF explores themes of ambition, loyalty, and revenge, making it a standout film in Indian cinema.\n\nThe story resonates with anyone who strives to overcome adversity and fulfill their dreams, reminding us that true power often comes at a cost.\n\n🎬 What are your favorite lessons or moments from KGF? Share your thoughts below!\n\n#Inspiration #Ambition #Cinema #Leadership #Storytelling #KGF2 #IndianFilm",
  'instagram': "💥 **KGF: The 

# Chain as a Runnable

In [3]:
# Task - 1 [Beautify Function]

def beautify(final_response: dict) -> dict:

    linkedin_response = final_response["branches"]["linkedin"]
    instagram_response = final_response["branches"]["instagram"]

    return {"linkedin": linkedin_response, "instagram": instagram_response}

beautify_runnable = RunnableLambda(beautify)

# TASK - 2 [Final Chain]

# final_chain

# Beautified Chain
beautified_chain = final_chain | beautify_runnable

beautified_chain.invoke("Pushpa")

[04/01/26 19:17:55] INFO     HTTP Request: POST                                                     ]8;id=768558;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=342612;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

[04/01/26 19:17:57] INFO     HTTP Request: POST                                                     ]8;id=924205;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=386481;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

[04/01/26 19:17:59] INFO     HTTP Request: POST                                                     ]8;id=2178;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=861598;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt                
                             -4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"                

{'linkedin': '🚩 **Leadership, Resilience, and the Spirit of Rising Above All Odds 🚩**\n\nJust watched **Pushpa: The Rise (2021)**—an action-packed Telugu film starring Allu Arjun—and it’s so much more than entertainment.\n\nIt’s the story of Pushpa Raj, a humble coolie from Andhra Pradesh who defies his social status to rise through a ruthless red sandalwood smuggling network. His journey isn’t just about crime; it’s a powerful metaphor for resilience, ambition, and the relentless pursuit of self-worth.\n\n🌳 Pushpa faces constant oppression but never bows down.  \n⚡ He leverages wit, courage, and quick thinking to outmaneuver rivals and overcome betrayal.  \n🌟 What truly stands out is his transformation—from an underdog to a formidable force, refusing to let societal labels define him.\n\n**Key Takeaways for Professionals:**\n- Don’t let your background limit your ambition.\n- Perseverance and adaptability are vital in the face of adversity.\n- True leadership is often forged in the to